## 5. Data Modelling
The models have been selected and saved, it will be run on the dataset and extract topic scores for each year, region and constituency and save this to csv files. 

In [1]:
# loading data
import pandas as pd
import pyarrow.parquet as pq

#for loading model and making models
from gensim import corpora
from gensim.models import LdaModel

#validation
import numpy as np


In [2]:
# import in the data from parquet files and csv
petitions_metadata_df = pd.read_csv('data\petitions_metadata.csv')

petitions_table = pq.read_table('data\petitions_processed_data.parquet')
countries_table = pq.read_table('data\petitions_country_data.parquet')
regions_table = pq.read_table('data\petitions_region_data.parquet')
constituencies_table = pq.read_table('data\petitions_constituency_data.parquet')

petitions_df = petitions_table.to_pandas()
countries_df = countries_table.to_pandas()
regions_df = regions_table.to_pandas()
constituencies_df = constituencies_table.to_pandas()

sample_df = petitions_df[petitions_df['created_at'].dt.year==2021]
processed_sample = sample_df['clean_text'].values

## Load model and test small sample

In [3]:
def load_model(year):
    """ 
    Function to load the relevant model for a given year
    :param year: integer between 2015-2025
    :output: dictionary, corpus and model for the inputted year
    """
    dictionary = corpora.Dictionary.load(f"models/model_{year}/dictionary.dict")
    corpus = corpora.MmCorpus(f"models/model_{year}/corpus.mm")
    model = LdaModel.load(f"models/model_{year}/model.gensim")

    return dictionary, corpus, model

def load_file(year):
    """ 
    Function to load the relevant filtered petition file for a given year
    :param year: integer between 2015-2025
    :output: file for the inputted year
    """
    file = petitions_df[petitions_df['created_at'].dt.year == year]

    return file

In [ ]:
# giving topics a generic label for the csv file
topic_labels = {
    0: "Topic 1",
    1: "Topic 2",
    2: "Topic 3",
    3: "Topic 4",
    4: "Topic 5", 
    5: "Topic 6", 
    6: "Topic 7",
    7: "Topic 8",
    8: "Topic 9",
    9: "Topic 10",
    10: "Topic 11",
    11: "Topic 12",
    12: "Topic 13",
    13: "Topic 14",
    14: "Topic 15",
    15: "Topic 16"
}

In [5]:
# will be examining how well the model performs for item 100
processed_sample[150]

array(['prioritise', 'receiving', 'vaccination', 'workplace', 'closed',
       'setting', 'remained', 'enabling', 'youngest', 'receiving',
       'socially', 'distance', 'nature', 'requires', 'contact',
       'intimate', 'wearing', 'ppe', 'recommended', 'afforded',
       'vaccination'], dtype=object)

In [6]:
# Check which topic our test document belongs to using the LDA Bag of Words model. Set document num: 
document_num = 150

dictionary, corpus, model = load_model(2021)

# looping through document to get the topics with highest probability
for index, score in sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, model.print_topic(index, 5)))


Score: 0.8022282719612122	 
Topic: 0.032*"quarantine" + 0.025*"vaccinated" + 0.023*"red" + 0.019*"vaccine" + 0.018*"mask"

Score: 0.15872056782245636	 
Topic: 0.017*"nursery" + 0.017*"cancer" + 0.016*"setting" + 0.013*"special" + 0.011*"visit"


Topic assigned is predominantly regarding quarantine and vaccine. Secondary is one regarding nurseries. The former definitely makes sense and there is some intuitive understanding to the second given the keywords selected.
## Testing topic assignment

In [7]:
def validate_corpus_file_alignment(file, corpus, sample_size=10):
    """
    Verify that corpus indices match file indices
    """
    if len(corpus) != len(file):
        print(f"WARNING: Corpus length ({len(corpus)}) != File length ({len(file)})")
        return False
    
    # Sample random documents and verify they make sense together
    import random
    random.seed(42)
    sample_indices = random.sample(range(min(len(corpus), len(file))), min(sample_size, len(corpus)))
    
    for idx in sample_indices:
        # Get original text
        original_text = file.iloc[idx]['clean_text']
        # Get corpus representation
        doc_bow = corpus[idx]
        
        print(f"\nDocument {idx}:")
        print(f"Original text (first 100 chars): {' '.join(original_text[:100])}")
        print(f"BOW length: {len(doc_bow)}")
        
    return True


In [8]:
def validate_topic_assignment(model, corpus, document_num, threshold=0.3):
    """
    Check if a document has a clear dominant topic
    :param threshold: minimum probability for dominant topic
    :return: dict with assignment quality metrics
    """
    topic_dist = model[corpus[document_num]]
    sorted_topics = sorted(topic_dist, key=lambda x: -x[1])
    
    dominant_prob = sorted_topics[0][1]
    second_prob = sorted_topics[1][1] if len(sorted_topics) > 1 else 0
    
    validation = {
        'document_num': document_num,
        'dominant_topic': sorted_topics[0][0],
        'dominant_prob': dominant_prob,
        'probability_gap': dominant_prob - second_prob,
        'is_clear_assignment': dominant_prob > threshold and (dominant_prob - second_prob) > 0.15,
        'entropy': -sum(p * np.log(p) if p > 0 else 0 for _, p in topic_dist)
    }
    
    return validation

In [9]:
possible_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# looping through years to get results
for year in possible_years:
    dictionary, corpus, model = load_model(year)
    file = load_file(year)

    print(f'{year}: corpus file alignment is {validate_corpus_file_alignment(file, corpus)}')

    unclear_assignments = []
    for doc_num in range(min(100, len(corpus))):  # Test first 100 docs
        val = validate_topic_assignment(model, corpus, doc_num)
        if not val['is_clear_assignment']:
            unclear_assignments.append(val)

    print(f"Documents with unclear assignments: {len(unclear_assignments)}")

2015: corpus file alignment is False
Documents with unclear assignments: 19

Document 912:
Original text (first 100 chars): release eu migrant insurance revealed minister hmrc refusing release active insurance issued eu migrant deem unhelpful eu renegotiation unacceptable released eu jonathan porte director institute requested electorate eu denied spirit fairness hiding electorate detrimental preferred
BOW length: 14

Document 204:
Original text (first 100 chars): litter clean overdue cheap simple clearing alarming litter detail promote creation nationwide voluntary litter clean publicly funded planning organising litter pick pick utilise volunteer organised controlled cleaned
BOW length: 16

Document 2253:
Original text (first 100 chars): exit eu june voting eu chose eu departure electorate exit eu regime negotiate eu interact eu voter knowing voting mandate negotiate eu exit final
BOW length: 11

Document 2006:
Original text (first 100 chars): disabled parking location ownership park

In [ ]:
def test_threshold_sensitivity(file, model, corpus, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """
    Test how threshold affects topic assignment counts
    """
    results = {}
    
    for threshold in thresholds:
        topic_counts = [0] * 16
        assigned_docs = 0
        
        for document_num in range(len(file)):
            if document_num < len(corpus):
                topic_probs = sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1])
                
                # Check if dominant topic exceeds threshold
                if topic_probs[0][1] > threshold:
                    topic_counts[topic_probs[0][0]] += 1
                    assigned_docs += 1
        
        results[threshold] = {
            'assigned_docs': assigned_docs,
            'unassigned_docs': len(file) - assigned_docs,
            'assignment_rate': assigned_docs / len(file),
            'topic_distribution': topic_counts
        }
    
    return pd.DataFrame(results).T

# Test threshold sensitivity on sample
dictionary, corpus, model = load_model(2023)
file = load_file(2023)

threshold_analysis = test_threshold_sensitivity(file, model, corpus)
print(threshold_analysis)

    assigned_docs unassigned_docs assignment_rate  \
0.3          3002              59        0.980725   
0.4          2709             352        0.885005   
0.5          2162             899        0.706305   
0.6          1565            1496        0.511271   
0.7          1104            1957        0.360666   

                                    topic_distribution  
0.3  [173, 158, 98, 131, 183, 157, 111, 172, 174, 2...  
0.4  [158, 144, 86, 118, 169, 142, 100, 157, 153, 1...  
0.5  [131, 118, 68, 94, 138, 121, 74, 124, 116, 150...  
0.6  [95, 71, 51, 75, 108, 83, 50, 87, 82, 105, 75,...  
0.7  [66, 54, 38, 61, 83, 50, 34, 56, 68, 69, 57, 7...  


In [11]:
# below code takes a petitions for a given topic and saves to a dataframe to then be examined further to understand topic groupings

def split_topic_petitions(file, topic_num, model, corpus):
    """ 
    Function to split petitions into given topics and save the results to a csv file
    :param file: filtered petition file for a given year
    :param topic_num: integer for the topic
    :param model: model for the given year
    :param corpus: corpus for the given year
    :output: file of petitions with the assigned topic
    """

    # setting up empty dataframe for petitions
    topic_df = pd.DataFrame(columns=['document', 'topic', 'topic prob', 'signature_count'])

    # loop through all documents
    for document_num in range(len(file)):
        # looping through topic probabilities to get the topics with highest probability
        for index, score in sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1]):
            # only want documents categorised as given topic
            if (index + 1 == topic_num) & (score>0.5):
                print("\nScore: {}\t \nTopic: {}".format(score, model.print_topic(index, 5)))
                # saving topics in dictionary 
                file_dict = {
                    'document': file.iloc[document_num, 6],
                    'topic': index + 1,
                    'topic prob': score,
                    'signature_count': file.iloc[document_num, 4]
                }
                #appending to end of dataframe
                topic_df.loc[len(topic_df)] = file_dict

    return topic_df

In [12]:
dictionary, corpus, model = load_model(2025)
petitions = load_file(2025)
extracted_df = split_topic_petitions(petitions, 5, model, corpus)
extracted_df.to_csv("data/2025_05.csv")


Score: 0.543110728263855	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.6038125157356262	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.8679035305976868	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.8782604932785034	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.8782604932785034	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.5528118014335632	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.6048294305801392	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.5283850431442261	 
Topic: 0.083*"sex" + 0.068*"equality" + 0.068*"trans" + 0.063*"offender" + 0.057*"gender"

Score: 0.8226719498634338	 
Topic: 0.083*"sex" +

So far, the breakdown of topics is ok but this doesn't tell us which topics have more significance or importance. This is just a division of all written petitions not taking into account so far that some petitions get a lot more signatures than others and so should be deemed more significant. For this, we need to weight categories based on signatures. To measure the impact of topics, each petition is counted and given a weight of one, to then measure the impact of each petition, this weight is multiplied by the number of signatures. A petition with 1000 signatures will add a score of 1000 to that topic.

In [ ]:
def find_top_topics(year_file, model, corpus):
    """
    This function loops through each document for a given year and weights each petition by its signature count.
    :param year_file: takes the pandas dataframe for a given year with documents listed
    :param model: model for given year
    :param corpus: corpus for given year
    :return: returns the list of topic weightings and a sorted list to find the top 3
    """
    errors = []
    processed_docs = 0
    # creates a list with an element for each topic to calculate topic totals
    topic_weighting = [0] * 16

    # loop through each petition
    for document_num in range(len(year_file)):
        try:
            # Check corpus bounds
            if document_num >= len(corpus):
                errors.append(f"Doc {document_num}: Out of corpus bounds")
                continue
            
            # Check for valid signature count
            num_signatures = year_file.iloc[document_num]['signature_count']
            if pd.isna(num_signatures) or num_signatures < 0:
                errors.append(f"Doc {document_num}: Invalid signature count")
                continue
            
            # Get topic probabilities
            topic_probs = model[corpus[document_num]]
            if not topic_probs:
                errors.append(f"Doc {document_num}: Empty topic distribution")
                continue
            
            # Calculate weightings
            for topic_id, prob in topic_probs:
                topic_weighting[topic_id] += num_signatures * prob
            
            processed_docs += 1
            
        except Exception as e:
            errors.append(f"Doc {document_num}: {str(e)}")
    
    sorted_topics = sorted(topic_weighting, reverse=True)

    # Log results
    print(f"\nProcessing Summary:")
    print(f"Total documents: {len(year_file)}")
    print(f"Successfully processed: {processed_docs}")
    print(f"Errors: {len(errors)}")
    
    if errors:
        print("\nFirst 5 errors:")
        for error in errors[:5]:
            print(f"  - {error}")
    
    return topic_weighting, sorted_topics, errors

In [ ]:
def find_area_top_topics(year_file, model, corpus):
    """
    This function loops through each document for a given year and area and weights each petition by its signature count. Difference
    between this and top_topic_count is in the use of the 'area_signature_count' column in the file as opposed to 'signature_count'
    :param year_file: takes the pandas dataframe for a given year with documents listed
    :param model: model for given year
    :param corpus: corpus for given year
    :return: returns the list of topic weightings and a sorted list to find the top 3
    """
    errors = []
    processed_docs = 0
    # creates a list with an element for each topic to calculate topic totals
    topic_weighting = [0] * 16

    # loop through each petition
    for document_num in range(len(year_file)):
        try:
            # Check corpus bounds
            if document_num >= len(corpus):
                errors.append(f"Doc {document_num}: Out of corpus bounds")
                continue
            
            # Check for valid signature count
            num_signatures = year_file.iloc[document_num]['area_signature_count']
            if pd.isna(num_signatures) or num_signatures < 0:
                errors.append(f"Doc {document_num}: Invalid signature count")
                continue
            
            # Get topic probabilities
            topic_probs = model[corpus[document_num]]
            if not topic_probs:
                errors.append(f"Doc {document_num}: Empty topic distribution")
                continue
            
            # Calculate weightings
            for topic_id, prob in topic_probs:
                topic_weighting[topic_id] += num_signatures * prob
            
            processed_docs += 1
            
        except Exception as e:
            errors.append(f"Doc {document_num}: {str(e)}")

    sorted_topics = sorted(topic_weighting, reverse=True)

    # Log results
    print(f"\nProcessing Summary:")
    print(f"Total documents: {len(year_file)}")
    print(f"Successfully processed: {processed_docs}")
    print(f"Errors: {len(errors)}")
    
    if errors:
        print("\nFirst 5 errors:")
        for error in errors[:5]:
            print(f"  - {error}")
    
    return topic_weighting, sorted_topics, errors

The above function will be tested on a small subset of 10 petitions first to see if the correct weighting has been given to each topic.

In [15]:
# testing on a small subset first
sample_df.head(10)

,petition_id,url,details,created_at,signature_count,parliament,text,clean_text
21882,565889,https://petition.parliament.uk/archived/petiti...,"On the 1st January 2021, the Government wisely...",2021-01-01 02:14:26.859000+00:00,1079,3,Abolish the 5% VAT applied to domestic gas and...,"[abolish, vat, applied, domestic, gas, electri..."
21883,565890,https://petition.parliament.uk/archived/petiti...,This is the best tv program ever made,2021-01-01 02:22:21.434000+00:00,21,3,"Bring back the Inbetweeners, Bring back the In...","[inbetweeners, inbetweeners, tv, program]"
21885,565898,https://petition.parliament.uk/archived/petiti...,Get out of lockdown and back to a normal econo...,2021-01-01 06:06:40.913000+00:00,21,3,Implement a 24/7 COVID-19 vaccination programm...,"[vaccination, programme, vaccinating, faster, ..."
21893,565914,https://petition.parliament.uk/archived/petiti...,Put the policies of opening schools in tier 4 ...,2021-01-01 10:31:23.226000+00:00,21,3,"Close early years settings in tier 4, Early ye...","[setting, tier, practitioner, nursery, centre,..."
21897,565925,https://petition.parliament.uk/archived/petiti...,Nigel Farage has been the single most successf...,2021-01-01 11:32:53.848000+00:00,21,3,"Give Nigel Farage a Knighthood, Give Nigel Far...","[nigel, farage, knighthood, nigel, farage, kni..."
21906,565953,https://petition.parliament.uk/archived/petiti...,Supertrawlers the length of 14 fishing boats s...,2021-01-01 14:17:50.103000+00:00,21,3,"BAN SUPER TRAWLERS FROM BRITISH WATERS, WE WAN...","[super, trawler, water, super, trawler, plunde..."
21907,565958,https://petition.parliament.uk/archived/petiti...,I call for all interactions between care home ...,2021-01-01 14:53:05.304000+00:00,46,3,Require all staff/resident interactions in car...,"[interaction, recorded, neglect, experiencing,..."
21911,565965,https://petition.parliament.uk/archived/petiti...,"Most political leaders, past and present, will...",2021-01-01 16:03:57.405000+00:00,21,3,"A gong for Nigel, Nigel Farage deserves recogn...","[gong, nigel, nigel, farage, deserves, recogni..."
21914,565970,https://petition.parliament.uk/archived/petiti...,Withdrawing from the single market is an act o...,2021-01-01 16:34:09.670000+00:00,16354,3,Apply to join the European Union Single Market...,"[join, eu, join, eu, withdrawing, sabotage, po..."
21919,565978,https://petition.parliament.uk/archived/petiti...,The Government should extend the date by which...,2021-01-01 17:25:30.756000+00:00,394,3,Extend EU Settlement Scheme for those affected...,"[eu, settlement, eu, settlement, started, dece..."


In [16]:
for document_num in range(len(sample_df.head(10))):
    print(sample_df.iloc[document_num].petition_id, sample_df.iloc[document_num].signature_count, 
          sorted(model[corpus[document_num]], key=lambda tup: -1*tup[1]))

565889 1079 [(7, 0.40980202), (9, 0.32034388), (11, 0.22063506)]
565890 21 [(0, 0.563272), (6, 0.2648466), (5, 0.12764475)]
565898 21 [(0, 0.3821328), (11, 0.3765039), (7, 0.19104113)]
565914 21 [(0, 0.35531232), (9, 0.30750808), (5, 0.21305154), (10, 0.083121985)]
565925 21 [(8, 0.90104485)]
565953 21 [(1, 0.88464963), (4, 0.073906526)]
565958 46 [(4, 0.54297817), (10, 0.40238887)]
565965 21 [(7, 0.51523715), (10, 0.14474435), (3, 0.13915803), (9, 0.13914323)]
565970 16354 [(6, 0.9226611)]
565978 394 [(3, 0.48512), (2, 0.431639)]


In [22]:
topic_weighting, sorted_topics, errors = find_top_topics(sample_df.head(10), model, corpus)
print(f"Errors: {errors}")
print("Topic weighting: ", topic_weighting)
# print the top 3 weighted topics
print("\nThe 3 most significant topics by signature count are: {}, " \
"{}, {}".format(topic_labels[topic_weighting.index(sorted_topics[0])], 
                            topic_labels[topic_weighting.index(sorted_topics[1])], 
                            topic_labels[topic_weighting.index(sorted_topics[2])]))


Processing Summary:
Total documents: 10
Successfully processed: 10
Errors: 0
Errors: []
Topic weighting:  [4.752140276134014, 95.94642683118582, 20.68749540299177, 1.7500000521540642, 12.96864091604948, 754.2896546348929, 11001.315411739051, 4693.498853974044, 188.8509485051036, 439.42759900540113, 1.7500000521540642, 28.941866151988506, 0, 0, 0, 0, 0, 0, 0, 0, 0]

The 3 most significant topics by signature count are: Topic 7, Topic 8, Topic 6


Test provides accurate results.


In [24]:
def generate_quality_report(year):
    """
    Comprehensive quality report for a given year's model
    """
    dictionary, corpus, model = load_model(year)
    file = load_file(year)
    
    report = {
        'year': year,
        'total_documents': len(file),
        'corpus_size': len(corpus),
        'alignment_check': len(corpus) == len(file),
    }
    
    # 1. Assignment clarity
    unclear_count = 0
    high_confidence_count = 0
    
    for doc_num in range(min(len(file), len(corpus))):
        val = validate_topic_assignment(model, corpus, doc_num)
        if not val['is_clear_assignment']:
            unclear_count += 1
        if val['dominant_prob'] > 0.7:
            high_confidence_count += 1
    
    report['unclear_assignments_pct'] = (unclear_count / len(corpus)) * 100
    report['high_confidence_pct'] = (high_confidence_count / len(corpus)) * 100
    
    # 2. Topic distribution balance
    topic_weighting, _, errors = find_top_topics(file, model, corpus)
    report['topic_distribution_std'] = np.std(topic_weighting)
    report['topic_distribution_cv'] = np.std(topic_weighting) / np.mean(topic_weighting)
    
    # 3. Coverage
    topic_weighting, _, errors = find_top_topics(file, model, corpus)
    top_3_coverage = sum(sorted(topic_weighting, reverse=True)[:3]) / sum(topic_weighting)
    report['top_3_topics_coverage_pct'] = top_3_coverage * 100
    
    return report

# Generate reports for all years
quality_reports = []
for year in possible_years:
    try:
        report = generate_quality_report(year)
        quality_reports.append(report)
        print(f"\n{year} Quality Report:")
        for key, value in report.items():
            print(f"  {key}: {value}")
    except Exception as e:
        print(f"Error processing {year}: {e}")

quality_df = pd.DataFrame(quality_reports)
quality_df.to_csv("data/model_quality_reports.csv", index=False)


Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

2015 Quality Report:
  year: 2015
  total_documents: 2037
  corpus_size: 2038
  alignment_check: False
  unclear_assignments_pct: 25.122669283611383
  high_confidence_pct: 51.37389597644749
  topic_distribution_std: 464882.75779685646
  topic_distribution_cv: 1.0790404649692764
  top_3_topics_coverage_pct: 43.249904520538756

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

2016 Quality Report:
  year: 2016
  total_documents: 5075
  corpus_size: 5075
  alignment_check: True
  unclear_assignments_pct: 26.482758620689655
  high_confidence_pct: 38.64039408866995
  topic_distribution_std: 1103039.6036740188
  topic_distribution_cv: 1.3038251749660676
  top_3_topics_coverage_pct: 51.96182540466611

Processing Summar


## Result extraction

In [27]:
# creating empty dataframe for the signature count calculations
top_topics_df = pd.DataFrame(columns=["Year", "Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", "Topic 5 Count", 
                                      "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", "Topic 9 Count", "Topic 10 Count", 
                                      "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", "Topic 15 Count", 
                                      "Topic 16 Count", "Top Topic"])

def top_topics_overall(file, model, corpus, year):
    """
    This function loops through each document and weights each petition by its signature count for each year or parliament.
    :param file: takes the pandas dataframe with documents listed
    :param parliament: integer (1-4)
    :param year: integer (2015-2025)
    :return: returns a dictionary contain information on that year, parliament and topic weightings
    """
    # sets up empty dictionary for top topics
    top_topics = {}

    filtered_petitions = file.loc[file['created_at'].dt.year == year]

    # calls the find_top_topics function defined earlier to find the topic weightings for that year or parliament
    topic_weighting, sorted_topics, errors = find_top_topics(filtered_petitions, model, corpus)

    #updates the empty dictionary to add the top topic values
    top_topics.update({
        "Year": year,
        "Topic 1 Count": topic_weighting[0],
        "Topic 2 Count": topic_weighting[1],
        "Topic 3 Count": topic_weighting[2],
        "Topic 4 Count": topic_weighting[3],
        "Topic 5 Count": topic_weighting[4],
        "Topic 6 Count": topic_weighting[5],
        "Topic 7 Count": topic_weighting[6],
        "Topic 8 Count": topic_weighting[7],
        "Topic 9 Count": topic_weighting[8],
        "Topic 10 Count": topic_weighting[9],
        "Topic 11 Count": topic_weighting[10],
        "Topic 12 Count": topic_weighting[11],
        "Topic 13 Count": topic_weighting[12],
        "Topic 14 Count": topic_weighting[13],
        "Topic 15 Count": topic_weighting[14],
        "Topic 16 Count": topic_weighting[15],
        "Top Topic": topic_labels[topic_weighting.index(sorted_topics[0])]
        })
        
    # printing some of the top topic results
    print("\nThe 3 most significant topics for {} by signature count are: " \
    "{}, {}, {}".format(year, topic_labels[topic_weighting.index(sorted_topics[0])], 
                                        topic_labels[topic_weighting.index(sorted_topics[1])], 
                                        topic_labels[topic_weighting.index(sorted_topics[2])]))
        
    return top_topics


In [28]:
possible_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# looping through years to get results
for year in possible_years:
    dictionary, corpus, model = load_model(year)
    file = load_file(year)

    topic_weighting, sorted_topics, errors = find_top_topics(file, model, corpus)

    top_topics_df.loc[len(top_topics_df)] = top_topics_overall(file, model, corpus, year)
    


Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

Processing Summary:
Total documents: 2037
Successfully processed: 2037
Errors: 0

The 3 most significant topics for 2015 by signature count are: Topic 9, Topic 11, Topic 5

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

Processing Summary:
Total documents: 5075
Successfully processed: 5075
Errors: 0

The 3 most significant topics for 2016 by signature count are: Topic 14, Topic 7, Topic 3

Processing Summary:
Total documents: 3330
Successfully processed: 3330
Errors: 0

Processing Summary:
Total documents: 3330
Successfully processed: 3330
Errors: 0

The 3 most significant topics for 2017 by signature count are: Topic 6, Topic 4, Topic 14

Processing Summary:
Total documents: 4287
Successfully processed: 4287
Errors: 0

Processing Summary:
Total documents: 4287
Successfully processed: 4287
Errors: 0

The 3 most significant topics for 2018 by signature count are: Topic 2

In [29]:
top_topics_df.to_csv("data/Top_topics.csv")

Next will be breaking up petitions into region to understand the geographical story:

In [30]:
def merge_petitions_region(file):
    """ 
    returns a file with petition signatures for a given region
    :param file: the filtered petition year file
    :output: a file containing region information along with signature count for each petition
    """
    # merge petitions with region and create region list
    petitions_by_region = file.merge(regions_df, left_on='petition_id', right_on='petition_id', how='right')
    petitions_by_region["area_signature_count"] = petitions_by_region["signature_count_y"]
    petitions_by_region["signature_count"] = petitions_by_region["signature_count_x"]
    
    return petitions_by_region

In [32]:
# creating list of region names to loop through
regions = regions_df['name'].unique()

# empty dataframe for signature count by topic by region
top_topics_region_df = pd.DataFrame(columns=["Region", "Year", "Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", 
                                             "Topic 5 Count", "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", "Topic 9 Count", 
                                             "Topic 10 Count", "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", 
                                             "Topic 15 Count", "Topic 16 Count", "Top Topic"])

def top_topics_by_region(file, region, year, model, corpus):
    """
    This function loops through each document for a given region and weights each petition by its signature count for each year or parliament.
    :param file: takes the pandas dataframe with documents listed
    :param region: string containg region information like 'North East'
    :param parliament: integer (1-4)
    :param year: integer (2015-2025)
    :return: returns a dictionary contain information on that region, year, parliament and topic weightings
    """
    # filter file by region and then year
    filtered_petitions = file.loc[file['name'] == region]
    filtered_petitions = filtered_petitions.loc[filtered_petitions['created_at'].dt.year == year]

    # calls the find_top_topics function defined earlier to find the topic weightings for that year or parliament
    topic_weighting, sorted_topics, errors = find_area_top_topics(filtered_petitions, model, corpus)

    top_topics = {
        "Region": region,
        "Year": year,
        "Topic 1 Count": topic_weighting[0],
        "Topic 2 Count": topic_weighting[1],
        "Topic 3 Count": topic_weighting[2],
        "Topic 4 Count": topic_weighting[3],
        "Topic 5 Count": topic_weighting[4],
        "Topic 6 Count": topic_weighting[5],
        "Topic 7 Count": topic_weighting[6],
        "Topic 8 Count": topic_weighting[7],
        "Topic 9 Count": topic_weighting[8],
        "Topic 10 Count": topic_weighting[9],
        "Topic 11 Count": topic_weighting[10],
        "Topic 12 Count": topic_weighting[11],
        "Topic 13 Count": topic_weighting[12],
        "Topic 14 Count": topic_weighting[13],
        "Topic 15 Count": topic_weighting[14],
        "Topic 16 Count": topic_weighting[15],
        "Top Topic": topic_labels[topic_weighting.index(sorted_topics[0])]
        }
    
    # print results fir top topics
    print("\nThe 3 most significant topics for {} in {} by signature count are: " \
    "{}, {}, {}".format(region, year, topic_labels[topic_weighting.index(sorted_topics[0])], 
                                        topic_labels[topic_weighting.index(sorted_topics[1])], 
                                        topic_labels[topic_weighting.index(sorted_topics[2])]))
    
    return top_topics

# looping through all years to extract results
for year in possible_years:
    # loading model and file and then merging
    dictionary, corpus, model = load_model(year)
    file = load_file(year)
    region_file = merge_petitions_region(file)
        
    # use region list as an index and for region in regions, filter for that region and apply the above, give the top 3 topics
    for region in regions:
        top_topics_region_df.loc[len(top_topics_region_df)] = top_topics_by_region(region_file, region, year, model, corpus)


Processing Summary:
Total documents: 1525
Successfully processed: 1525
Errors: 0

The 3 most significant topics for North East in 2015 by signature count are: Topic 6, Topic 4, Topic 12

Processing Summary:
Total documents: 1828
Successfully processed: 1828
Errors: 0

The 3 most significant topics for North West in 2015 by signature count are: Topic 9, Topic 2, Topic 8

Processing Summary:
Total documents: 1701
Successfully processed: 1701
Errors: 0

The 3 most significant topics for Yorkshire and The Humber in 2015 by signature count are: Topic 8, Topic 11, Topic 12

Processing Summary:
Total documents: 1706
Successfully processed: 1706
Errors: 0

The 3 most significant topics for East Midlands in 2015 by signature count are: Topic 2, Topic 8, Topic 9

Processing Summary:
Total documents: 1832
Successfully processed: 1832
Errors: 0

The 3 most significant topics for West Midlands in 2015 by signature count are: Topic 5, Topic 12, Topic 2

Processing Summary:
Total documents: 1814
Suc

In [33]:
top_topics_region_df.to_csv("data/Top_topics_by_region.csv")

In [35]:
def merge_consituency_petitions(file):
    """ 
    returns a file with petition signatures for a given constituency
    :param file: the filtered petition year file
    :output: a file containing constituency information along with signature count for each petition
    """
    # merge petitions with constituency and creating constituency list
    petitions_by_constituency = file.merge(constituencies_df, left_on='petition_id', right_on='petition_id', how='right')
    petitions_by_constituency["area_signature_count"] = petitions_by_constituency["signature_count_y"]
    petitions_by_constituency["signature_count"] = petitions_by_constituency["signature_count_x"]

    return petitions_by_constituency

# list of constituency names to loop through
constituencies = constituencies_df['name'].unique()

# empty dataframe to calculate signature count by topic by constituency and year
top_topics_con_df = pd.DataFrame(columns=["Constituency", "Year", "Topic 1 Count", "Topic 2 Count", "Topic 3 Count", "Topic 4 Count", 
                                             "Topic 5 Count", "Topic 6 Count", "Topic 7 Count", "Topic 8 Count", "Topic 9 Count", 
                                             "Topic 10 Count", "Topic 11 Count", "Topic 12 Count", "Topic 13 Count", "Topic 14 Count", 
                                             "Topic 15 Count", "Topic 16 Count", "Top Topic"])

def top_topics_by_con(file, constituency, year, model, corpus):
    """
    This function loops through each document for a given constituency and weights each petition by its signature count for each year or parliament.
    :param file: takes the pandas dataframe with documents listed
    :param constituency: string containing region information like 'Bristol West'
    :param parliament: integer (1-4)
    :return: returns a dictionary contain information on that constituency, parliament and topic weightings
    """
    top_topics = {}

    # filtering file by constituency name and year
    filtered_petitions = file.loc[file['name'] == constituency]
    filtered_petitions = filtered_petitions.loc[filtered_petitions['created_at'].dt.year == year]

    # finds topic weightings for the filtered petitions
    topic_weighting, sorted_topics, errors = find_top_topics(filtered_petitions, model, corpus)

    # saves information on that constituency in a dictionary
    top_topics.update({
        "Constituency": constituency,
        "Year": year,
        "Topic 1 Count": topic_weighting[0],
        "Topic 2 Count": topic_weighting[1],
        "Topic 3 Count": sorted_topics[2],
        "Topic 4 Count": sorted_topics[3],
        "Topic 5 Count": sorted_topics[4],
        "Topic 6 Count": sorted_topics[5],
        "Topic 7 Count": sorted_topics[6],
        "Topic 8 Count": sorted_topics[7],
        "Topic 9 Count": sorted_topics[8],
        "Topic 10 Count": topic_weighting[9],
        "Topic 11 Count": topic_weighting[10],
        "Topic 12 Count": topic_weighting[11],
        "Topic 13 Count": topic_weighting[12],
        "Topic 14 Count": topic_weighting[13],
        "Topic 15 Count": topic_weighting[14],
        "Topic 16 Count": topic_weighting[15],
        "Top Topic": topic_labels[topic_weighting.index(sorted_topics[0])]
        })
    
    # prints top topic results
    print("\nThe 3 most significant topics for {} in {} by signature count are: " \
    "{}, {}, {}".format(constituency, year, topic_labels[topic_weighting.index(sorted_topics[0])], 
                                        topic_labels[topic_weighting.index(sorted_topics[1])], 
                                        topic_labels[topic_weighting.index(sorted_topics[2])]))
    
    return top_topics

# to keep track o how many consituencies have been looked up (more than 650 as constituency boundaries have changed over time)
con_count = 0

# looping through years to extract all results
for year in possible_years:
    # load model and file and then merge
    dictionary, corpus, model = load_model(year)
    file = load_file(year)
    con_file = merge_consituency_petitions(file)

    # use constituency list as an index and for constituency in constituencies, filter for that constituency and apply 
    # the above, give the top 3 topics and append topic weightings to df
    for constituency in constituencies:

        # find top topics by constituenc and add dictionary returned to end of dataframe
        top_topics_con_df.loc[len(top_topics_con_df)] = top_topics_by_con(con_file, constituency, year, model, corpus)
        
        # keeping track of how many constituencies have been processed
        con_count += 1
        print('Constituency count:', con_count)


Processing Summary:
Total documents: 708
Successfully processed: 708
Errors: 0

The 3 most significant topics for Aldershot in 2015 by signature count are: Topic 6, Topic 12, Topic 10
Constituency count: 1

Processing Summary:
Total documents: 639
Successfully processed: 639
Errors: 0

The 3 most significant topics for Altrincham and Sale West in 2015 by signature count are: Topic 7, Topic 11, Topic 4
Constituency count: 2

Processing Summary:
Total documents: 633
Successfully processed: 633
Errors: 0

The 3 most significant topics for Amber Valley in 2015 by signature count are: Topic 6, Topic 7, Topic 2
Constituency count: 3

Processing Summary:
Total documents: 617
Successfully processed: 617
Errors: 0

The 3 most significant topics for Ashfield in 2015 by signature count are: Topic 9, Topic 11, Topic 6
Constituency count: 4

Processing Summary:
Total documents: 607
Successfully processed: 607
Errors: 0

The 3 most significant topics for Ashton-under-Lyne in 2015 by signature count

In [36]:
top_topics_con_df.to_csv("data/Top_topics_by_constituency.csv")

For reference for some reading on similar topics which have assisted in setting aims for the project and understanding the model, see below.

### Bibliography
- Vidgen B, Yasseri T. What, when and where of petitions submitted to the UK government during a time of chaos. Policy sciences. 2020;53(3):535-557. doi:10.1007/s11077-020-09395-y
- Bicalho P, Pita M, Pedrosa G, Lacerda A, Pappa GL. A general framework to expand short text for topic modeling. Information sciences. 2017;393:66-81. doi:10.1016/j.ins.2017.02.007
- Hagen L. Content analysis of e-petitions with topic modeling: How to train and evaluate LDA models? Information processing & management. 2018;54(6):1292-1307. doi:10.1016/j.ipm.2018.05.006